# บทเรียน 02 - สำรวจ Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** คือกรอบงานรวมสำหรับสร้างเอเย่นต์ AI ซึ่งมีสถาปัตยกรรมที่สะอาดและประกอบด้วยบล็อกหลักสี่ส่วน:

- **Client** – เชื่อมต่อกับจุดเชื่อมต่อของโมเดล AI และจัดการการสื่อสาร
- **Agent** – ห่อหุ้ม client ด้วยคำสั่งและนิยามเครื่องมือ
- **Tools** – ขยายความสามารถของเอเย่นต์ด้วยฟังก์ชันที่กำหนดเองที่โมเดลสามารถเรียกใช้ได้
- **Session** – รักษาประวัติการสนทนาสำหรับปฏิสัมพันธ์หลายรอบ

ในบทเรียนนี้ เราจะสร้าง **เอเย่นต์จองท่องเที่ยว** ที่ตรวจสอบความพร้อมของจุดหมายโดยใช้แนวคิดเหล่านี้


## การตั้งค่า


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ทำความเข้าใจกับสถาปัตยกรรมกรอบงาน Agent

กรอบงาน Microsoft Agent ใช้สถาปัตยกรรมแบบมีชั้นดังนี้:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `FoundryChatClient` เชื่อมต่อกับการปรับใช้ Azure OpenAI ซึ่งจัดการการพิสูจน์ตัวตน การจัดรูปแบบคำขอ และการแยกวิเคราะห์การตอบสนอง
2. **Agent** – สร้างจาก client ผ่าน `provider.create_agent()` เอเจนต์รวมการเข้าถึงโมเดลกับคำสั่ง (prompt ระบบ) และเครื่องมือ
3. **Tools** – ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่งเอเจนต์สามารถเรียกใช้เพื่อดำเนินการหรือนำข้อมูลมาใช้
4. **Session** – ออบเจ็กต์ `AgentSession` (สร้างผ่าน `agent.create_session()`) ที่เก็บประวัติการสนทนา ทำให้สามารถโต้ตอบแบบหลายรอบที่เอเจนต์จดจำบริบทก่อนหน้าได้

เรามาสร้างแต่ละชั้นทีละขั้นตอนกันเถอะ


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## การเพิ่มเครื่องมือด้วย @tool Decorator

เครื่องมือช่วยให้ตัวแทนสามารถดำเนินการได้มากกว่าการสร้างข้อความ `@tool` decorator จะแปลงฟังก์ชัน Python ธรรมดาให้กลายเป็นสิ่งที่ตัวแทนสามารถเรียกใช้ได้

ประเด็นสำคัญ:
- ใช้ `Annotated[type, "description"]` เพื่อให้โมเดลเข้าใจพารามิเตอร์แต่ละตัว
- docstring จะกลายเป็นคำอธิบายเครื่องมือที่โมเดลเห็น
- `approval_mode="never_require"` หมายถึงเครื่องมือจะทำงานโดยอัตโนมัติโดยไม่ต้องยืนยันจากผู้ใช้


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## การสร้างเอเย่นต์ด้วยเครื่องมือ

ตอนนี้เรารวมไคลเอนต์ คำแนะนำ และเครื่องมือเข้าเป็นเอเย่นต์เดียวกัน `instructions` ทำหน้าที่เป็นพรอมต์ระบบ — กำหนดบุคลิกและพฤติกรรมของเอเย่นต์


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## การสนทนาหลายรอบด้วย Sessions

`AgentSession` (สร้างผ่าน `agent.create_session()`) จะติดตามข้อความทั้งหมดในการสนทนา โดยการส่ง session เดียวกันไปยังแต่ละการเรียก `agent.run()` เอเจนต์จะสามารถเข้าถึงประวัติการสนทนาทั้งหมดและสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าได้

เราส่ง `tools=[check_destination_availability]` เพื่อให้เอเจนต์สามารถเรียกตัวตรวจสอบความพร้อมของเราในแต่ละรอบได้


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## สรุป

ในบทเรียนนี้คุณได้สำรวจเสาหลักสี่ประการของ Microsoft Agent Framework:

| แนวคิด | สิ่งที่คุณได้เรียนรู้ |
|---------|------------------|
| **Client** | `FoundryChatClient` เชื่อมต่อกับ Azure OpenAI ด้วยการยืนยันตัวตนแบบใช้ข้อมูลรับรอง |
| **Agent** | `provider.create_agent()` รวมการเชื่อมต่อโมเดลกับคำสั่งและชื่อ |
| **Tools** | การใช้ตัวตกแต่ง `@tool` เพื่อเปิดเผยฟังก์ชัน Python ให้ตัวแทนเรียกใช้งาน |
| **Session** | `agent.create_session()` รักษาประวัติการสนทนาข้ามการโต้ตอบหลายรอบ |

บล็อกพื้นฐานเหล่านี้ประกอบกันเพื่อสร้างตัวแทนที่สามารถสนทนาแบบธรรมชาติ เรียกใช้งานฟังก์ชันภายนอก และรักษาบริบท — ซึ่งเป็นรากฐานสำหรับรูปแบบตัวแทนที่ซับซ้อนยิ่งขึ้นในบทเรียนต่อไป


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
